# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library. The dataset is described using the [Croissant](https://mlcommons.github.io/croissant/) schema, includes multiple record sets and fields, and is appropriate for exploration and analysis with Python.

### Dataset Source
The dataset metadata is provided via the Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access top-level metadata (not subscriptable)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")

# Optionally print available record sets
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"- {rs.id}")

## 2. Data Overview
Examine the dataset's record sets, their associated fields, and IDs. This helps you identify which portions of data are available for extraction and their structure.

In [ ]:
# List all record sets and their fields by @id
print("Record sets and their fields:")
for record_set in dataset.record_sets:
    print(f"RecordSet: {record_set.id} ({record_set.name})")
    for field in record_set.fields:
        ftype = getattr(field, 'data_type', None) or getattr(field, 'type', None) or ''
        print(f"  Field: {field.id} ({field.name})   Type: {ftype}")
    print()

If you want to preview actual records, use `.records` with a specific record set `@id`.

In [ ]:
# Show an example of extracting records for a particular record set.
# Replace below with a real @id from the previous cell's output.

# List all available record set @id values:
available_record_sets = [rs.id for rs in dataset.record_sets]
example_record_set_id = available_record_sets[0] if available_record_sets else None

if example_record_set_id:
    print(f"Showing up to 3 records for record set {example_record_set_id}:")
    
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i >= 2:
            break
else:
    print("No record sets available in this dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for further analysis. Always reference record sets and fields using their `@id`.

In [ ]:
# Collect DataFrames for each record set, indexed by their @id
dataframes = {}
for record_set in dataset.record_sets:
    rs_id = record_set.id
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set: {rs_id}")

# Choose one record set for further operations
main_record_set_id = None
for rs_id, df in dataframes.items():
    main_record_set_id = rs_id
    break

if main_record_set_id:
    print(f"\nColumns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

## 4. Exploratory Data Analysis (EDA)
Demonstrate various data processing steps:
- Filtering numeric fields
- Normalizing data
- Grouping by categorical fields
*Reference all columns/fields by their `@id`.*

In [ ]:
# --- Setup: Find usable numeric field and group field by @id ---
import numpy as np

df = dataframes.get(main_record_set_id)
if df is not None:
    # Find a numeric field by @id (heuristic: type or values look numeric)
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        # Heuristic: looks for columns that have numeric dtype after conversion or contain 'coeff'/'loglik'/'value' in id
        if df[col].dtype.kind in ['i', 'f']:
            numeric_field_id = col
            break
        try:
            numeric = pd.to_numeric(df[col], errors='coerce')
            if numeric.notna().sum() > 0 and (('coeff' in col.lower()) or ('loglik' in col.lower()) or (numeric.notna().mean() > 0.5)):
                numeric_field_id = col
                break
        except Exception:
            pass
    # Find a group field (categorical/id/variable)
    for col in df.columns:
        if 'variable' in col.lower() or 'group' in col.lower() or 'ward' in col.lower():
            group_field_id = col
            if group_field_id != numeric_field_id:
                break

    print(f"Numeric field selected for filtering and normalization: {numeric_field_id}")
    if group_field_id:
        print(f"Group field selected: {group_field_id}")
    else:
        # If column names are just integers (ordered regression outputs), pick the first string/categorical
        for col in df.columns:
            if df[col].dtype == object and col != numeric_field_id:
                group_field_id = col
                break

    # Coerce the numeric column to floats (for normalization/filtering)
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Filter for values greater than a threshold
    threshold = np.nanpercentile(df[numeric_field_id], 75) if df[numeric_field_id].notna().sum() > 0 else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[numeric_field_id + "_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())

    # Grouping by a categorical field, if available
    if group_field_id and group_field_id in filtered_df.columns:
        group_stats = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(group_stats.head())
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize the distribution of a numeric variable, and compare groups if a group field is available. All field references are by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load the FAIR² dataset, inspect its structure using Croissant `@id`-based references, and perform initial explorations and analysis in pandas. All interactions referenced entities by their `@id` for maximum robustness and reproducibility.

- The dataset describes ordered logistic regression results on factors predicting knowledge adoption in pastoral rangeland management.
- We explored available record sets, selected fields by `@id`, demonstrated filtering/numeric normalization, and visualized data distributions.

Feel free to further investigate field relationships, run statistical tests, or connect this workflow with your own models or downstream analyses.